# HHT IF traces on CWT scalogram

Overlays per-mode HHT instantaneous frequency (IF) traces on the CWT scalogram.
Both representations share the same time and frequency axes, so MVMD mode placement
can be compared directly against CWT energy density.

Each scatter point sits at `(t, IF[mode, ch, t])` and is coloured by the corresponding
envelope amplitude.  Only in-band samples `[F_MIN_HHT, F_MAX_HHT]` are shown.

Produces lognorm and linear figures per channel.

In [1]:
from pathlib import Path

from lib.fs.project_config import REPO_ROOT
from lib.fs.get_script_name import get_name

import h5py
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

import plot_config  # noqa: F401

In [2]:
# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
TR = 0.8  # seconds

F_MIN_HHT = 0.010  # Hz — in-band floor (SLOW_BANDS)
F_MAX_HHT = 0.250  # Hz — in-band ceiling

F_MIN_CWT = 0.008  # Hz — CWT scale grid
F_MAX_CWT = 0.500  # Hz

subject_id  = "sub-NDARINVAG388HJL"
file_suffix = "_task-restAP_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.h5"

preprocessing_dir = Path("/Volumes/work/bids_processed_consolidated_data")
h5_path = preprocessing_dir / subject_id / f"{subject_id}{file_suffix}"

channels_to_plot = [21]

out_dir = REPO_ROOT / "figures" / get_name()
out_dir.mkdir(parents=True, exist_ok=True)

print(f"h5_path: {h5_path}")
print(f"out_dir: {out_dir}")

h5_path: /Volumes/work/bids_processed_consolidated_data/sub-NDARINVAG388HJL/sub-NDARINVAG388HJL_task-restAP_run-01_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold.h5
out_dir: /Users/ipeglin/Git/masters-thesis-supplementary/figures/plot_hht_cwt_comparison


In [ ]:
# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
with h5py.File(h5_path, "r") as f:
    hht_grp   = f["04hht/full_run_std"]
    inst_freq = hht_grp["instantaneous_frequency"][:]  # [n_modes, n_ch, n_t]
    envelope  = hht_grp["envelope"][:]                 # [n_modes, n_ch, n_t]
    scalogram = f["03cwt/full_run_std"][:]             # [n_ch, n_scales, n_t]

n_modes, n_channels, n_timepoints = inst_freq.shape
time_axis = np.arange(n_timepoints) * TR

n_scales = scalogram.shape[1]
CWT_FREQ_AXIS = F_MIN_CWT * (F_MAX_CWT / F_MIN_CWT) ** (
    np.arange(n_scales) / (n_scales - 1)
)  # log-spaced ascending

print(f"inst_freq:  {inst_freq.shape}  [n_modes, n_ch, n_t]")
print(f"envelope:   {envelope.shape}")
print(f"scalogram:  {scalogram.shape}  [n_ch, n_scales, n_t]")

In [ ]:
# ---------------------------------------------------------------------------
# Plot: HHT IF traces overlaid on CWT scalogram
# ---------------------------------------------------------------------------
CWT_CMAP = "gray"
IF_CMAP  = "hot_r"  # contrasts against gray CWT background


def plot_if_on_cwt(ch, log_norm):
    norm_label = "lognorm" if log_norm else "linear"
    cwt_2d = scalogram[ch]  # [n_scales, n_t]

    pos = cwt_2d[cwt_2d > 0]
    if log_norm and pos.size > 0:
        cwt_norm = mcolors.LogNorm(
            vmin=np.percentile(pos, 1), vmax=np.percentile(pos, 99)
        )
    else:
        cwt_norm = mcolors.Normalize(
            vmin=cwt_2d.min(), vmax=np.percentile(cwt_2d, 99)
        )

    fig, ax = plt.subplots()

    # CWT background
    ax.pcolormesh(
        time_axis, CWT_FREQ_AXIS, cwt_2d,
        shading="auto", cmap=CWT_CMAP, norm=cwt_norm,
        linewidth=0, antialiased=False, edgecolors="none", rasterized=True,
    )

    # Collect all in-band IF points across modes
    t_pts, if_pts, env_pts = [], [], []
    for m in range(n_modes):
        if_m  = inst_freq[m, ch, :]
        env_m = envelope[m, ch, :]
        mask  = (if_m >= F_MIN_HHT) & (if_m < F_MAX_HHT)
        t_pts.append(time_axis[mask])
        if_pts.append(if_m[mask])
        env_pts.append(env_m[mask])

    t_pts   = np.concatenate(t_pts)
    if_pts  = np.concatenate(if_pts)
    env_pts = np.concatenate(env_pts)

    env_max  = env_pts.max()
    env_norm = env_pts / env_max if env_max > 0 else env_pts

    sc = ax.scatter(
        t_pts, if_pts,
        c=env_norm, cmap=IF_CMAP,
        s=0.5, linewidths=0, alpha=0.6, rasterized=True,
    )
    fig.colorbar(sc, ax=ax, label="HHT envelope (normalised)")

    ax.set_yscale("log")
    ax.set_ylim(F_MIN_HHT, F_MAX_HHT)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_title(f"CWT + HHT IF traces  |  {subject_id}  Ch {ch}  ({norm_label})")
    return fig


for ch in channels_to_plot:
    for log_norm in (True, False):
        norm_label = "lognorm" if log_norm else "linear"
        fig = plot_if_on_cwt(ch, log_norm)
        out_path = out_dir / f"{subject_id}_ch-{ch}_cwt_hht_if_{norm_label}.pdf"
        fig.savefig(out_path, format="pdf")
        print(f"Saved: {out_path}")
        plt.show()
        plt.close(fig)